In [ ]:
# ==================== STAGE 1: COLAB SETUP & MS-COCO DOWNLOAD ====================
import os
from google.colab import drive

# 1. Mount Google Drive (we will use this at the very end to save the model and tokenizer)
drive.mount('/content/drive')

# 2. Create local folder for MS-COCO dataset
COCO_DIR = "/content/coco"
os.makedirs(COCO_DIR, exist_ok=True)

# 3. Install fast downloader (aria2) and unzip tool
!apt-get update -qq && apt-get install -y -qq aria2 unzip

# 4. Download and extract Captions Annotations (241 MB)
!aria2c -x 16 -s 16 -d {COCO_DIR} http://images.cocodataset.org/annotations/annotations_trainval2017.zip
!unzip -q {COCO_DIR}/annotations_trainval2017.zip -d {COCO_DIR}
!rm {COCO_DIR}/annotations_trainval2017.zip

# 5. Download and extract Train2017 Images (18 GB)
!aria2c -x 16 -s 16 -d {COCO_DIR} http://images.cocodataset.org/zips/train2017.zip
!unzip -q {COCO_DIR}/train2017.zip -d {COCO_DIR}
!rm {COCO_DIR}/train2017.zip

# 6. Download and extract Val2017 Images (1 GB - required for validation dataset)
!aria2c -x 16 -s 16 -d {COCO_DIR} http://images.cocodataset.org/zips/val2017.zip
!unzip -q {COCO_DIR}/val2017.zip -d {COCO_DIR}
!rm {COCO_DIR}/val2017.zip

print("[SUCCESS] MS-COCO Train, Val, and Annotations are ready!")

In [ ]:
# ==================== STAGE 2A: TOKENIZER FUNCTIONS ====================
import os
import json
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Hyperparameters
MAX_VOCAB_SIZE = 10000
MAX_SEQ_LENGTH = 40

def impage_Caption_relation(img_dir, captions_json_path):
    """
    Reads the MS-COCO JSON file and matches each image path with its caption.
    Adds <start> and <end> tags to every caption.
    """
    with open(captions_json_path, "r", encoding="utf-8") as f:
        coco_Dataset = json.load(f)

    # Map image ID to its full file path
    imageid_To_path = {
        img["id"]: os.path.join(img_dir, img["file_name"])
        for img in coco_Dataset["images"]
    }

    img_paths = []
    captions  = []

    # Pair each caption annotation with its corresponding image path
    for ann in coco_Dataset["annotations"]:
        img_id = ann["image_id"]
        if img_id in imageid_To_path:
            img_paths.append(imageid_To_path[img_id])
            caption_text = f"<start> {ann['caption'].strip()} <end>"
            captions.append(caption_text)

    return img_paths, captions


def build_tokenizer(train_captions, max_words=MAX_VOCAB_SIZE):
    """
    Fits a brand-new Keras Tokenizer in memory on the training captions.
    Keeps '<' and '>' so <start> and <end> are not removed.
    """
    custom_filters = '!"#$%&()*+,-./:;=?@[\\]^_`{|}~\t\n'

    tokenizer = Tokenizer(
        num_words=max_words,
        oov_token='<unk>',
        filters=custom_filters,
        lower=True
    )
    tokenizer.fit_on_texts(train_captions)
    return tokenizer


def tokenize_and_pad(captions, tokenizer, max_len=MAX_SEQ_LENGTH):
    """
    Converts text captions into integer lists and pads them with zeros at the end.
    """
    sequences = tokenizer.texts_to_sequences(captions)
    padded_seqs = pad_sequences(
        sequences,
        maxlen=max_len,
        padding='post',
        truncating='post'
    )
    return padded_seqs


def decode_tokenizer_sequence(token_ids, tokenizer):
    """
    Converts a list of integer token IDs back into a readable English sentence.
    """
    words = []
    for idx in token_ids:
        if idx == 0:  # Skip padding zeros
            continue
        word = tokenizer.index_word.get(idx, '<unk>')
        if word == '<start>':
            continue
        if word == '<end>':
            break
        words.append(word)
    return " ".join(words)

In [ ]:
# ==================== STAGE 2B: EXECUTE TOKENIZER ====================
TRAIN_IMG_DIR = f"{COCO_DIR}/train2017"
TRAIN_JSON_PATH = f"{COCO_DIR}/annotations/captions_train2017.json"
VAL_IMG_DIR = f"{COCO_DIR}/val2017"
VAL_JSON_PATH = f"{COCO_DIR}/annotations/captions_val2017.json"

# 1. Load image paths and captions for Train and Val
train_img_paths, train_captions = impage_Caption_relation(TRAIN_IMG_DIR, TRAIN_JSON_PATH)
val_img_paths, val_captions = impage_Caption_relation(VAL_IMG_DIR, VAL_JSON_PATH)

# 2. Fit Tokenizer on training captions in memory
tokenizer = build_tokenizer(train_captions, max_words=MAX_VOCAB_SIZE)

# 3. Convert captions to padded integer sequences
padded_train_seqs = tokenize_and_pad(train_captions, tokenizer)
padded_val_seqs = tokenize_and_pad(val_captions, tokenizer)

print(f"Total Train Pairs: {len(train_img_paths)} | Total Val Pairs: {len(val_img_paths)}")

In [ ]:
# ==================== STAGE 3A: DATA PIPELINE FUNCTIONS ====================
import tensorflow as tf
from tensorflow import keras

BATCH_SIZE = 512
IMG_SIZE = 224

def preprocess_image_keras(img_path):
    """Reads an image from disk, resizes it to 224x224, and formats pixels for ResNet50."""
    raw_bytes = tf.io.read_file(img_path)
    img = tf.image.decode_image(raw_bytes, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])  # decode_image doesn't set static shape; resize needs it
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = keras.applications.resnet50.preprocess_input(img)
    return img


def prepare_caption_sample(img_path, padded_seq):
    """
    Loads the image and shifts the caption by 1 word for Teacher Forcing:
    decoder_input  = [<start>, a, dog, <end>, 0]
    decoder_target = [a, dog, <end>, 0, 0]
    """
    img_tensor = preprocess_image_keras(img_path)
    seq = tf.cast(padded_seq, tf.int32)

    decoder_input = seq[:-1]
    decoder_target = seq[1:]
    
    sample_weight = tf.cast(tf.not_equal(decoder_target, 0), tf.float32)

    return (img_tensor, decoder_input), decoder_target, sample_weight


def build_datasets(train_paths, train_seqs, val_paths, val_seqs):
    """Creates fast tf.data.Dataset pipelines for training and validation."""
    # 1. Build Training Dataset
    train_ds = tf.data.Dataset.from_tensor_slices((train_paths, train_seqs))
    train_ds = train_ds.shuffle(buffer_size=2000, seed=42)
    train_ds = train_ds.map(prepare_caption_sample, num_parallel_calls=tf.data.AUTOTUNE)
    train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=False)

    # 2. Build Validation Dataset
    val_ds = tf.data.Dataset.from_tensor_slices((val_paths, val_seqs))
    val_ds = val_ds.map(prepare_caption_sample, num_parallel_calls=tf.data.AUTOTUNE)
    val_ds = val_ds.batch(BATCH_SIZE, drop_remainder=False)

    # 3. Prefetch batches in the background for faster GPU training
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds

In [ ]:
# ==================== STAGE 3B: EXECUTE DATA PIPELINE ====================
train_ds, val_ds = build_datasets(
    train_img_paths,
    padded_train_seqs,
    val_img_paths,
    padded_val_seqs
)

print(f"Train batches: {len(train_ds)} | Val batches: {len(val_ds)}")

In [ ]:
# ==================== STAGE 4: BUILD MODEL ARCHITECTURE ====================
from tensorflow.keras.layers import *
from tensorflow.keras.models import Model
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.utils import plot_model

MAX_WORDS = 10000

# Define two sets of inputs:
# inputA = 1 Image (224x224 with 3 RGB color channels)
# inputB = Text Sequence (None allows 39 words during training and 1-by-1 words during testing)
inputA = Input(shape=(224, 224, 3), name="image_input")
inputB = Input(shape=(None,), name="text_input")

# Load pretrained ResNet50 CNN and freeze its weights
base_cnn = ResNet50(weights="imagenet", include_top=False, pooling="avg")
base_cnn.trainable = False

# First branch operates on the first input (Image)
x1 = base_cnn(inputA)
x2 = Dropout(0.3)(x1)

# Compress the 2048 image features into 256 units for the LSTM memory states (h and c)
x_h = Dense(256, activation="relu")(x2)
x_c = Dense(256, activation="relu")(x2)

# Second branch operates on the second input (Text Sequence)
y1 = Embedding(input_dim=MAX_WORDS, output_dim=256, mask_zero=True)(inputB)
y2 = Dropout(0.3)(y1)

# Combine the output of the two branches inside the LSTM
# y2 (text) flows into the LSTM, while [x_h, x_c] (image) initializes its memory
combined = LSTM(256, return_sequences=True)(y2, initial_state=[x_h, x_c])

# Apply Dropout and a Dense layer to predict the next word across all 10,000 words
z1 = Dropout(0.3)(combined)
z2 = Dense(MAX_WORDS, activation="linear", name="predictions")(z1)

# Build the final model accepting [Image, Text] and outputting word scores
LSTM_Model = Model(inputs=[inputA, inputB], outputs=z2)

# Print model summary
LSTM_Model.summary()

In [ ]:
# ==================== STAGE 5A: COMPILE MODEL & DEFINE TRAIN FUNCTION ====================
import tensorflow as tf

SAVE_DIR = "/content/drive/MyDrive/LiveSight_Keras_Saved"
os.makedirs(SAVE_DIR, exist_ok=True)   

# Custom Metric: returns 2D matches [batch_size, 39] so it works with Embedding(mask_zero=True)
def masked_accuracy(y_true, y_pred):
    mask = tf.math.not_equal(y_true, 0)
    predicted_ids = tf.cast(tf.argmax(y_pred, axis=-1), tf.int32)
    y_true_cast = tf.cast(y_true, tf.int32)
    matches = tf.math.equal(y_true_cast, predicted_ids)
    matches = tf.logical_and(mask, matches)
    return tf.cast(matches, tf.float32)

# Compile the model
LSTM_Model.compile(
    optimizer='adam',
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=[masked_accuracy]
)

# NEW: callbacks for checkpointing + early stopping
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath="/content/drive/MyDrive/LiveSight_Keras_Saved/best_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
]

def train_model(model, train_dataset, val_dataset, epochs=15, callbacks=None):
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=epochs,
        callbacks=callbacks,
        verbose=1
    )
    return history

In [ ]:
# ==================== STAGE 5B: EXECUTE TRAINING ====================
History_1 = train_model(LSTM_Model, train_ds, val_ds, epochs=15, callbacks=callbacks)

In [ ]:
# ==================== STAGE 5B: EXECUTE TRAINING (SMALL SUBSET) ====================
# .take(60) runs only 60 batches (~30k captions) per epoch for fast training
small_train_ds = train_ds.take(180)
small_val_ds = val_ds.take(10)

History_1 = train_model(LSTM_Model, small_train_ds, small_val_ds, epochs=15, callbacks=callbacks)

In [ ]:
# ==================== STAGE 6A: SINGLE CAPTION GENERATOR FUNCTION ====================
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from PIL import Image

START_TOKEN_ID = tokenizer.word_index['<start>']
END_TOKEN_ID = tokenizer.word_index['<end>']

def generate_1_caption(model, img_tensor, tokenizer_obj, max_len=40):
    """
    Generates 1 caption for an image by picking the highest-scoring word at each step.
    """
    # Add batch dimension to image -> Shape: (1, 224, 224, 3)
    img_batch = tf.expand_dims(img_tensor, axis=0)

    # Start the sequence with the <start> token
    caption_seq = [START_TOKEN_ID]

    for _ in range(max_len):
        # Convert current word list to a tensor -> Shape: (1, current_length)
        seq_input = tf.constant([caption_seq], dtype=tf.int32)

        # Predict word scores from the model
        logits = model([img_batch, seq_input], training=False)

        # Take the scores for the very last word predicted -> Shape: (vocab_size,)
        last_word_logits = logits[0, -1, :].numpy()

        # Block <PAD> (0) and <unk> so the model only picks real words
        last_word_logits[0] = -1e9
        if '<unk>' in tokenizer_obj.word_index:
            last_word_logits[tokenizer_obj.word_index['<unk>']] = -1e9

        # Pick the word ID with the highest score
        next_word_id = int(np.argmax(last_word_logits))
        caption_seq.append(next_word_id)

        # Stop generating once the model predicts <end>
        if next_word_id == END_TOKEN_ID:
            break

    # Convert the predicted token IDs back into a clean English sentence
    return decode_tokenizer_sequence(caption_seq, tokenizer_obj)

In [ ]:
# ==================== STAGE 6B: EXECUTE TEST & DISPLAY ====================
test_idx = 108
test_img_path = train_img_paths[test_idx]
preprocessed_img = preprocess_image_keras(test_img_path)

# Generate 1 caption
predicted_caption = generate_1_caption(LSTM_Model, preprocessed_img, tokenizer)

# Show the test image and print the generated caption
plt.figure(figsize=(6, 6))
plt.imshow(Image.open(test_img_path))
plt.axis("off")
plt.show()

print(f"Generated Caption: {predicted_caption}")

In [ ]:
# ==================== STAGE 7: SAVE MODEL (.keras) & TOKENIZER (.pkl) ====================
import os
import pickle

SAVE_DIR = "/content/drive/MyDrive/LiveSight_Keras_Saved"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Save the trained model in .keras format
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "LSTM_Model.keras")
LSTM_Model.save(MODEL_SAVE_PATH)

# 2. Save the tokenizer in .pkl format
TOKENIZER_SAVE_PATH = os.path.join(SAVE_DIR, "tokenizer.pkl")
with open(TOKENIZER_SAVE_PATH, "wb") as f:
    pickle.dump(tokenizer, f)

print(f"Model saved to     : {MODEL_SAVE_PATH}")
print(f"Tokenizer saved to : {TOKENIZER_SAVE_PATH}")